# 16 · Multi-Hop Retrieval (LangGraph)

Use hop-1 evidence to invent a better hop-2 query — with an explicit hop cap.

**Analogy handbook:** [multi-hop](../retriever-analogy-handbook.html#multi-hop)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


##### multihop retriever practical

### Learning: MULTI-HOP RETRIEVAL PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `MULTI-HOP RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# MULTI-HOP RETRIEVAL PRACTICAL
# LangChain + LangGraph
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL PACKAGES
# ------------------------------------------------------------

# %pip install -U langchain langchain-openai langgraph


# ============================================================
# 2. IMPORTS
# ============================================================

from typing import TypedDict, List

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import (
    StateGraph,
    START,
    END,
)


### Learning: CREATE BASE RETRIEVER

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE BASE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:


# ============================================================
# 3. CREATE BASE RETRIEVER
# ============================================================

# Existing Chroma vector_store from previous practical

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Base retriever created.")


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created.")


### Learning: DEFINE GRAPH STATE

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `DEFINE GRAPH STATE` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:

# ============================================================
# 5. DEFINE GRAPH STATE
# ============================================================

class MultiHopState(TypedDict):

    original_query: str

    hop1_query: str

    hop1_documents: List[Document]

    hop1_summary: str

    hop2_query: str

    hop2_documents: List[Document]

    final_answer: str


### Learning: HOP 1 RETRIEVAL

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: HOP 1 RETRIEVAL.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 6. HOP 1 RETRIEVAL
# ============================================================

def hop1_retrieve(
    state: MultiHopState
):

    query = state["original_query"]

    print("\n" + "=" * 100)
    print("HOP 1 QUERY:")
    print(query)

    documents = retriever.invoke(
        query
    )

    print("\nHOP 1 DOCUMENTS:")

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\nDocument {i}"
        )

        print(
            "Page:",
            doc.metadata.get(
                "paper_page"
            )
        )

        print(
            doc.page_content[:500]
        )

    return {
        "hop1_query": query,
        "hop1_documents": documents
    }


### Learning: SUMMARIZE HOP 1 EVIDENCE

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `SUMMARIZE HOP 1 EVIDENCE` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 7. SUMMARIZE HOP 1 EVIDENCE
# ============================================================

hop1_summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are helping with multi-hop retrieval.

Read the retrieved context and extract only the
information useful for determining what should
be searched in the next retrieval hop.

Do not answer the final user question yet.
"""
        ),
        (
            "human",
            """
Original question:

{query}


Retrieved context:

{context}
"""
        )
    ]
)


### Learning: def summarize_hop1(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def summarize_hop1(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:

def summarize_hop1(
    state: MultiHopState
):

    context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop1_documents"
        ]
    )

    response = llm.invoke(
        hop1_summary_prompt.format_messages(
            query=state[
                "original_query"
            ],
            context=context
        )
    )

    summary = response.content

    print("\n" + "=" * 100)
    print("HOP 1 SUMMARY:")
    print(summary)

    return {
        "hop1_summary": summary
    }



### Learning: GENERATE SECOND-HOP QUERY

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `GENERATE SECOND-HOP QUERY` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 8. GENERATE SECOND-HOP QUERY
# ============================================================

next_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are generating the next search query
for a multi-hop retrieval system.

Based on:

1. the original user question
2. the evidence retrieved in Hop 1

generate ONE new search query that retrieves
the missing information needed to answer the
original question.

Rules:

- Do not answer the final question.
- Return only the new search query.
- The query must be standalone.
- Use information discovered in Hop 1.
"""
        ),
        (
            "human",
            """
Original question:

{original_query}


Hop 1 evidence:

{hop1_summary}
"""
        )
    ]
)


### Learning: def generate_hop2_query(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def generate_hop2_query(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
def generate_hop2_query(
    state: MultiHopState
):

    response = llm.invoke(
        next_query_prompt.format_messages(
            original_query=state[
                "original_query"
            ],
            hop1_summary=state[
                "hop1_summary"
            ]
        )
    )

    hop2_query = (
        response.content.strip()
    )

    print("\n" + "=" * 100)
    print("GENERATED HOP 2 QUERY:")
    print(hop2_query)

    return {
        "hop2_query": hop2_query
    }

### Learning: HOP 2 RETRIEVAL

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: HOP 2 RETRIEVAL.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. HOP 2 RETRIEVAL
# ============================================================

def hop2_retrieve(
    state: MultiHopState
):

    query = state[
        "hop2_query"
    ]

    documents = retriever.invoke(
        query
    )

    print("\n" + "=" * 100)
    print("HOP 2 DOCUMENTS:")

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\nDocument {i}"
        )

        print(
            "Page:",
            doc.metadata.get(
                "paper_page"
            )
        )

        print(
            doc.page_content[:500]
        )

    return {
        "hop2_documents": documents
    }


### Learning: GENERATE FINAL ANSWER

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `GENERATE FINAL ANSWER` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 10. GENERATE FINAL ANSWER
# ============================================================

final_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Answer the user's question using only the
evidence retrieved during Hop 1 and Hop 2.

If the evidence is insufficient, say that the
retrieved documents are insufficient.

Do not invent information.
"""
        ),
        (
            "human",
            """
Original question:

{query}


Hop 1 evidence:

{hop1_context}


Hop 2 evidence:

{hop2_context}
"""
        )
    ]
)

### Learning: def generate_final_answer(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def generate_final_answer(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
def generate_final_answer(
    state: MultiHopState
):

    hop1_context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop1_documents"
        ]
    )

    hop2_context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop2_documents"
        ]
    )

    response = llm.invoke(
        final_answer_prompt.format_messages(
            query=state[
                "original_query"
            ],
            hop1_context=hop1_context,
            hop2_context=hop2_context
        )
    )

    final_answer = (
        response.content
    )

    print("\n" + "=" * 100)
    print("FINAL ANSWER:")
    print(final_answer)

    return {
        "final_answer":
            final_answer
    }


### Learning: BUILD LANGGRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `BUILD LANGGRAPH` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 11. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(
    MultiHopState
)


# Add nodes

builder.add_node(
    "hop1_retrieve",
    hop1_retrieve
)

builder.add_node(
    "summarize_hop1",
    summarize_hop1
)

builder.add_node(
    "generate_hop2_query",
    generate_hop2_query
)

builder.add_node(
    "hop2_retrieve",
    hop2_retrieve
)

builder.add_node(
    "generate_final_answer",
    generate_final_answer
)


### Learning: CONNECT GRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `CONNECT GRAPH` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 12. CONNECT GRAPH
# ============================================================

builder.add_edge(
    START,
    "hop1_retrieve"
)

builder.add_edge(
    "hop1_retrieve",
    "summarize_hop1"
)

builder.add_edge(
    "summarize_hop1",
    "generate_hop2_query"
)

builder.add_edge(
    "generate_hop2_query",
    "hop2_retrieve"
)

builder.add_edge(
    "hop2_retrieve",
    "generate_final_answer"
)

builder.add_edge(
    "generate_final_answer",
    END
)


### Learning: COMPILE GRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Executes retrieval/generation for: COMPILE GRAPH.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 13. COMPILE GRAPH
# ============================================================

multi_hop_graph = (
    builder.compile()
)

print(
    "\nMulti-Hop Retrieval Graph created successfully."
)


# ============================================================
# 14. TEST QUERY
# ============================================================

query = (
    "How was Llama 2-Chat aligned with human preferences "
    "and what role did reward models play in that process?"
)


result = multi_hop_graph.invoke(
    {
        "original_query": query
    }
)


### Learning: FINAL OUTPUT

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `FINAL OUTPUT` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 15. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("ORIGINAL QUERY:")
print(
    result["original_query"]
)

print("\n" + "=" * 100)
print("HOP 1 QUERY:")
print(
    result["hop1_query"]
)

print("\n" + "=" * 100)
print("HOP 1 SUMMARY:")
print(
    result["hop1_summary"]
)

print("\n" + "=" * 100)
print("HOP 2 QUERY:")
print(
    result["hop2_query"]
)

print("\n" + "=" * 100)
print("FINAL ANSWER:")
print(
    result["final_answer"]
)

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 16. FINAL CONCEPTUAL FLOW
# ============================================================

"""
MULTI-HOP RETRIEVAL

Original User Query
        ↓
Hop 1 Retrieval
        ↓
Retrieve first evidence
        ↓
Analyze / summarize evidence
        ↓
Generate next query
        ↓
Hop 2 Retrieval
        ↓
Retrieve additional evidence
        ↓
Combine Hop 1 + Hop 2 evidence
        ↓
LLM
        ↓
Final Answer
"""


print(
    "\nMulti-Hop Retrieval practical completed successfully."
)